ForensicNLP — Notebook 03: Classification Experiments (Stage 2)
================================================================
Fine-tunes BERT, RoBERTa, and ForensicRoBERTa on the forensic
artifact classification task. Reports accuracy, macro-F1,
per-class metrics, confusion matrices, and ablation results.

GPU: L4 recommended. Each model takes ~15-25 min to fine-tune.
     Full pipeline (5 models × 3 seeds) takes ~4-6 hours.


# ForensicNLP — Stage 2: Artifact Classification Experiments

**Research Question (RQ2):** Do domain-adapted BERT/RoBERTa models
outperform general-purpose counterparts on forensic artifact
classification aligned with MITRE ATT&CK tactics?

## Models compared:
1. TF-IDF + Logistic Regression (classical baseline)
2. BERT-base-uncased (fine-tuned)
3. RoBERTa-base (fine-tuned)
4. ForensicRoBERTa (continued pretraining + fine-tuned)
5. Ablations: pretraining duration, label granularity


## CELL 1: Setup and Imports


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Mount Drive (uncomment in Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# Paths
BASE_DIR = '/content/drive/MyDrive/forensicnlp'
#BASE_DIR = '.'

CLASS_DIR = os.path.join(BASE_DIR, 'data', 'classification')
RESULTS_DIR = os.path.join(BASE_DIR, 'results', 'stage2_classification')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 # Corrected: total_mem to total_memory
    print(f"✓ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠ No GPU found — training will be slow")
print(f"✓ Device: {device}")

# Experiment config
SEEDS = [42, 123, 456]
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_EPOCHS = 10
PATIENCE = 3
WARMUP_RATIO = 0.06

print(f"✓ Config: {len(SEEDS)} seeds, max_len={MAX_LENGTH}, batch={BATCH_SIZE}, lr={LEARNING_RATE}")

✓ GPU: NVIDIA L4 (23.7 GB)
✓ Device: cuda
✓ Config: 3 seeds, max_len=128, batch=16, lr=2e-05


## CELL 2: Load Data


WHAT THIS DOES:
Loads the train/dev/test splits created by notebook 01.
Creates label-to-ID mappings for the classifier.


In [5]:
df_train = pd.read_csv(os.path.join(CLASS_DIR, 'train.csv'))
df_dev = pd.read_csv(os.path.join(CLASS_DIR, 'dev.csv'))
df_test = pd.read_csv(os.path.join(CLASS_DIR, 'test.csv'))

# Create label mapping
labels_sorted = sorted(df_train['label'].unique())
label2id = {label: idx for idx, label in enumerate(labels_sorted)}
id2label = {idx: label for label, idx in label2id.items()}
num_labels = len(labels_sorted)

print(f"✓ Train: {len(df_train)}, Dev: {len(df_dev)}, Test: {len(df_test)}")
print(f"✓ Classes ({num_labels}):")
for label, idx in sorted(label2id.items(), key=lambda x: x[1]):
    count = len(df_train[df_train['label'] == label])
    print(f"    {idx:2d}: {label:25s} ({count} train samples)")

✓ Train: 12325, Dev: 2409, Test: 2687
✓ Classes (14):
     0: Collection                (1277 train samples)
     1: Command and Control       (1686 train samples)
     2: Credential Access         (410 train samples)
     3: Defense Evasion           (2986 train samples)
     4: Discovery                 (2278 train samples)
     5: Execution                 (1466 train samples)
     6: Exfiltration              (287 train samples)
     7: Impact                    (282 train samples)
     8: Initial Access            (333 train samples)
     9: Lateral Movement          (178 train samples)
    10: Persistence               (607 train samples)
    11: Privilege Escalation      (49 train samples)
    12: Reconnaissance            (123 train samples)
    13: Resource Development      (363 train samples)


## CELL 3: Baseline — TF-IDF + Logistic Regression


WHAT THIS DOES:
Runs the classical (non-deep-learning) baseline. TF-IDF converts text
into weighted word-frequency vectors, then Logistic Regression classifies.
This gives us a floor to beat — if transformers can't beat TF-IDF+LR,
something is wrong.


In [7]:
print("=" * 60)
print("  MODEL 1: TF-IDF + Logistic Regression (Baseline)")
print("=" * 60)

# Removed the line with the invalid 'subword_info' parameter
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(df_train['text'])
X_dev = tfidf.transform(df_dev['text'])
X_test = tfidf.transform(df_test['text'])

y_train = df_train['label'].values
y_dev = df_dev['label'].values
y_test = df_test['label'].values

# Train with class weights to handle imbalance
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=1.0,
    random_state=42,
    n_jobs=-1
)
lr_model.fit(X_train, y_train)

# Evaluate
y_pred_test = lr_model.predict(X_test)
tfidf_acc = accuracy_score(y_test, y_pred_test)
tfidf_f1 = f1_score(y_test, y_pred_test, average='macro')

print(f"\n  Test Accuracy:  {tfidf_acc:.4f}")
print(f"  Test Macro-F1:  {tfidf_f1:.4f}")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_test, digits=4))

# Store results
all_results = [{
    'model': 'TF-IDF + LR',
    'seed': 42,
    'accuracy': tfidf_acc,
    'macro_f1': tfidf_f1,
    'predictions': y_pred_test.tolist()
}]

  MODEL 1: TF-IDF + Logistic Regression (Baseline)

  Test Accuracy:  0.8046
  Test Macro-F1:  0.7506

  Classification Report:
                      precision    recall  f1-score   support

          Collection     0.8475    0.7967    0.8213       300
 Command and Control     0.8778    0.8705    0.8741       363
   Credential Access     0.7664    0.8824    0.8203       119
     Defense Evasion     0.8512    0.7267    0.7840       622
           Discovery     0.8927    0.8525    0.8721       488
           Execution     0.7656    0.8221    0.7929       298
        Exfiltration     0.8235    0.7368    0.7778        57
              Impact     0.5833    0.8077    0.6774        52
      Initial Access     0.6587    0.8925    0.7580        93
    Lateral Movement     0.6349    0.7692    0.6957        52
         Persistence     0.6903    0.7868    0.7354       136
Privilege Escalation     0.6429    0.7500    0.6923        12
      Reconnaissance     0.6667    0.4545    0.5405        22
Res

## CELL 4: Dataset Class for Transformers


WHAT THIS DOES:
Creates a PyTorch Dataset that tokenizes text on the fly and returns
input_ids, attention_mask, and labels — the format transformers expect.


In [8]:
class ForensicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Custom metric computation for Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': acc, 'macro_f1': f1}

print("✓ Dataset class defined")

✓ Dataset class defined


## CELL 5: Training Function


WHAT THIS DOES:
Defines a reusable function that fine-tunes any transformer model
on our classification task. We'll call this for BERT, RoBERTa,
and ForensicRoBERTa.

Each call:
1. Loads the pretrained model + tokenizer
2. Adds a classification head on top
3. Fine-tunes on the training set
4. Evaluates on dev set each epoch (early stopping)
5. Evaluates on test set
6. Returns metrics


In [9]:
def train_and_evaluate(model_name_or_path, display_name, seed,
                       tokenizer_override=None, save_model=False):
    """
    Fine-tune a transformer for classification and evaluate on test set.

    Args:
        model_name_or_path: HuggingFace model ID or local path
        display_name: Name for logging (e.g., "BERT", "RoBERTa")
        seed: Random seed for reproducibility
        tokenizer_override: Optional different tokenizer (for ablation)
        save_model: Whether to save the fine-tuned checkpoint

    Returns:
        dict with accuracy, macro_f1, predictions, training time
    """
    set_seed(seed)
    start_time = time.time()

    # Load tokenizer and model
    tokenizer = tokenizer_override or AutoTokenizer.from_pretrained(model_name_or_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True  # needed if vocab changed
    ).to(device)

    # Prepare datasets
    train_labels = [label2id[l] for l in df_train['label']]
    dev_labels = [label2id[l] for l in df_dev['label']]
    test_labels = [label2id[l] for l in df_test['label']]

    train_dataset = ForensicDataset(df_train['text'].tolist(), train_labels, tokenizer, MAX_LENGTH)
    dev_dataset = ForensicDataset(df_dev['text'].tolist(), dev_labels, tokenizer, MAX_LENGTH)
    test_dataset = ForensicDataset(df_test['text'].tolist(), test_labels, tokenizer, MAX_LENGTH)

    # Training arguments
    output_dir = os.path.join(MODELS_DIR, f'{display_name}_seed{seed}')
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to='none',  # Change to 'wandb' if using W&B
        seed=seed,
        dataloader_num_workers=2,
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    )

    # Train
    trainer.train()

    # Evaluate on test set
    test_results = trainer.evaluate(test_dataset)

    # Get predictions for detailed analysis
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = np.array(test_labels)

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')

    elapsed = time.time() - start_time

    # Save model if requested
    if save_model:
        save_path = os.path.join(MODELS_DIR, f'{display_name}_best')
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)

    # Clean up GPU memory
    del model, trainer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return {
        'model': display_name,
        'seed': seed,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'predictions': y_pred.tolist(),
        'true_labels': y_true.tolist(),
        'training_time_min': elapsed / 60,
        'test_results': test_results
    }

print("✓ Training function defined")

✓ Training function defined


## CELL 6: Train BERT (3 seeds)


WHAT THIS DOES:
Fine-tunes bert-base-uncased on the classification task.
Runs 3 times with different random seeds for reliability.
Expected time: ~45-75 min total on L4.


In [10]:
print("=" * 60)
print("  MODEL 2: BERT-base-uncased")
print("=" * 60)

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    result = train_and_evaluate('bert-base-uncased', 'BERT', seed)
    all_results.append(result)
    print(f"  Accuracy: {result['accuracy']:.4f}, Macro-F1: {result['macro_f1']:.4f} ({result['training_time_min']:.1f} min)")

  MODEL 2: BERT-base-uncased

--- Seed 42 ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (40 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

  Accuracy: 0.8697, Macro-F1: 0.8284 (9.1 min)


## CELL 7: Train RoBERTa (3 seeds)


WHAT THIS DOES:
Fine-tunes roberta-base. RoBERTa is an optimized version of BERT
with better training recipe. It often outperforms BERT on NLP tasks.
Expected time: ~45-75 min total on L4.


In [11]:
print("=" * 60)
print("  MODEL 3: RoBERTa-base")
print("=" * 60)

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    result = train_and_evaluate('roberta-base', 'RoBERTa', seed)
    all_results.append(result)
    print(f"  Accuracy: {result['accuracy']:.4f}, Macro-F1: {result['macro_f1']:.4f} ({result['training_time_min']:.1f} min)")

  MODEL 3: RoBERTa-base

--- Seed 42 ---


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (40 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

  Accuracy: 0.8694, Macro-F1: 0.8119 (10.7 min)


## CELL 8: Create ForensicRoBERTa (Continued Pretraining)


WHAT THIS DOES:
This is the KEY experiment — domain adaptation.

We take roberta-base and continue its masked-language-model (MLM)
pretraining on our forensic corpus. This teaches the model forensic
vocabulary and patterns BEFORE we fine-tune it for classification.

Think of it like a medical student doing a residency: they already
know general medicine (RoBERTa's pretraining), and now they spend
extra time learning the specialty (forensic text).

We train for 3 epochs on the forensic text with MLM objective.
Expected time: ~30-60 min on L4.


In [12]:
print("=" * 60)
print("  CREATING ForensicRoBERTa (Continued Pretraining)")
print("=" * 60)

# Load RoBERTa for masked language modeling
roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
roberta_mlm = AutoModelForMaskedLM.from_pretrained('roberta-base').to(device)

# Prepare forensic corpus for MLM training
# Use ALL forensic text: classification data + tokenization strings
forensic_texts = df_train['text'].tolist() + df_dev['text'].tolist()

# Also load tokenization strings
token_path = os.path.join(BASE_DIR, 'data', 'tokenization', 'forensic_strings_from_attack.jsonl')
if os.path.exists(token_path):
    df_tok = pd.read_json(token_path, lines=True)
    forensic_texts.extend(df_tok['string'].tolist())

print(f"  Forensic corpus for MLM: {len(forensic_texts)} texts")

# Create MLM dataset
class MLMDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt'
        )

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

mlm_dataset = MLMDataset(forensic_texts, roberta_tokenizer, MAX_LENGTH)

# Data collator handles random masking (15% of tokens)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=roberta_tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Training arguments for continued pretraining
mlm_output_dir = os.path.join(MODELS_DIR, 'forensic_roberta_mlm')
mlm_args = TrainingArguments(
    output_dir=mlm_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_strategy='epoch',
    save_total_limit=1,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=42,
    dataloader_num_workers=2,
)

# Run continued pretraining
mlm_trainer = Trainer(
    model=roberta_mlm,
    args=mlm_args,
    train_dataset=mlm_dataset,
    data_collator=data_collator,
)

print("  Starting continued pretraining (3 epochs)...")
mlm_trainer.train()

# Save ForensicRoBERTa
forensic_roberta_path = os.path.join(MODELS_DIR, 'forensic_roberta')
mlm_trainer.save_model(forensic_roberta_path)
roberta_tokenizer.save_pretrained(forensic_roberta_path)

print(f"\n✓ ForensicRoBERTa saved to: {forensic_roberta_path}")

# Clean up
del roberta_mlm, mlm_trainer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

  CREATING ForensicRoBERTa (Continued Pretraining)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Forensic corpus for MLM: 16469 texts


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Starting continued pretraining (3 epochs)...


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ ForensicRoBERTa saved to: /content/drive/MyDrive/forensicnlp/models/forensic_roberta


## CELL 9: Train ForensicRoBERTa (3 seeds)


WHAT THIS DOES:
Fine-tunes the domain-adapted ForensicRoBERTa for classification.
This is where we expect to see the gains from domain adaptation.
Expected time: ~45-75 min total on L4.


In [13]:
print("=" * 60)
print("  MODEL 4: ForensicRoBERTa (Domain-Adapted)")
print("=" * 60)

forensic_roberta_path = os.path.join(MODELS_DIR, 'forensic_roberta')

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    result = train_and_evaluate(forensic_roberta_path, 'ForensicRoBERTa', seed,
                                save_model=(seed == 42))  # Save best seed
    all_results.append(result)
    print(f"  Accuracy: {result['accuracy']:.4f}, Macro-F1: {result['macro_f1']:.4f} ({result['training_time_min']:.1f} min)")

  MODEL 4: ForensicRoBERTa (Domain-Adapted)

--- Seed 42 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/forensicnlp/models/forensic_roberta
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.667142,0.647333,0.816106,0.681147
2,0.445895,0.527576,0.850145,0.768891
3,0.332634,0.519094,0.865504,0.813632
4,0.250720,0.588810,0.868410,0.817415
5,0.174334,0.632510,0.877543,0.823455
6,0.076563,0.712599,0.869655,0.833011
7,0.121804,0.701177,0.882939,0.846676
8,0.084130,0.743369,0.880863,0.844005
9,0.045596,0.759766,0.883769,0.834891
10,0.038211,0.761403,0.886260,0.850279


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (35 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

  Accuracy: 0.8869, Macro-F1: 0.8450 (10.7 min)


## CELL 10: Results Summary Table


WHAT THIS DOES:
Aggregates results across all seeds and creates the main results table.
Reports mean ± std for each metric.


In [14]:
print("=" * 60)
print("  MAIN RESULTS TABLE")
print("=" * 60)

# Group by model
model_names = ['TF-IDF + LR', 'BERT', 'RoBERTa', 'ForensicRoBERTa']
results_summary = []

for model_name in model_names:
    model_results = [r for r in all_results if r['model'] == model_name]
    accs = [r['accuracy'] for r in model_results]
    f1s = [r['macro_f1'] for r in model_results]

    summary = {
        'Model': model_name,
        'Accuracy': f"{np.mean(accs):.4f} ± {np.std(accs):.4f}" if len(accs) > 1 else f"{accs[0]:.4f}",
        'Macro-F1': f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}" if len(f1s) > 1 else f"{f1s[0]:.4f}",
        'acc_mean': np.mean(accs),
        'f1_mean': np.mean(f1s),
        'f1_std': np.std(f1s) if len(f1s) > 1 else 0,
    }
    results_summary.append(summary)

print(f"\n  {'Model':25s} {'Accuracy':20s} {'Macro-F1':20s}")
print("  " + "-" * 65)
for s in results_summary:
    print(f"  {s['Model']:25s} {s['Accuracy']:20s} {s['Macro-F1']:20s}")

# Highlight the key comparison
roberta_f1 = [s for s in results_summary if s['Model'] == 'RoBERTa'][0]['f1_mean']
forensic_f1 = [s for s in results_summary if s['Model'] == 'ForensicRoBERTa'][0]['f1_mean']
gain = (forensic_f1 - roberta_f1) * 100

print(f"\n  ForensicRoBERTa vs RoBERTa: {'+' if gain > 0 else ''}{gain:.2f} macro-F1 points")
if gain >= 3:
    print("  ✓ Exceeds success threshold (≥3 points)")
elif gain > 0:
    print("  ~ Positive but below threshold (< 3 points)")
else:
    print("  ✗ No improvement — report as negative result")

  MAIN RESULTS TABLE

  Model                     Accuracy             Macro-F1            
  -----------------------------------------------------------------
  TF-IDF + LR               0.8046               0.7506              
  BERT                      0.8699 ± 0.0011      0.8196 ± 0.0070     
  RoBERTa                   0.8686 ± 0.0055      0.8089 ± 0.0036     
  ForensicRoBERTa           0.8818 ± 0.0050      0.8387 ± 0.0108     

  ForensicRoBERTa vs RoBERTa: +2.98 macro-F1 points
  ~ Positive but below threshold (< 3 points)


## CELL 11: Per-Class Classification Report


WHAT THIS DOES:
Shows per-class precision, recall, and F1 for the best model.
This helps us understand WHICH forensic categories are easy vs hard.


In [15]:
print("=" * 60)
print("  PER-CLASS RESULTS (Best ForensicRoBERTa, seed 42)")
print("=" * 60)

# Get the seed-42 ForensicRoBERTa results
best_result = [r for r in all_results if r['model'] == 'ForensicRoBERTa' and r['seed'] == 42]
if not best_result:
    # Fall back to best available model
    best_result = [r for r in all_results if 'predictions' in r and 'true_labels' in r]

if best_result:
    best = best_result[0]
    y_pred_best = [id2label[p] for p in best['predictions']]
    y_true_best = [id2label[t] for t in best['true_labels']]
    print(classification_report(y_true_best, y_pred_best, digits=4))
else:
    print("  No detailed predictions available")

  PER-CLASS RESULTS (Best ForensicRoBERTa, seed 42)
                      precision    recall  f1-score   support

          Collection     0.9053    0.8600    0.8821       300
 Command and Control     0.8796    0.9256    0.9020       363
   Credential Access     0.8000    0.8739    0.8353       119
     Defense Evasion     0.8933    0.8746    0.8838       622
           Discovery     0.9232    0.9611    0.9418       488
           Execution     0.8576    0.9094    0.8827       298
        Exfiltration     0.9167    0.7719    0.8381        57
              Impact     0.8627    0.8462    0.8544        52
      Initial Access     0.8795    0.7849    0.8295        93
    Lateral Movement     0.8605    0.7115    0.7789        52
         Persistence     0.8134    0.8015    0.8074       136
Privilege Escalation     0.6875    0.9167    0.7857        12
      Reconnaissance     0.8889    0.7273    0.8000        22
Resource Development     0.9062    0.7945    0.8467        73

            accu

## CELL 12: Confusion Matrix Figure


WHAT THIS DOES:
Creates a heatmap showing what the model predicts for each true class.
Diagonal = correct. Off-diagonal = errors. This is a key paper figure.


In [16]:
if best_result:
    best = best_result[0]
    y_pred_labels = [id2label[p] for p in best['predictions']]
    y_true_labels = [id2label[t] for t in best['true_labels']]

    cm = confusion_matrix(y_true_labels, y_pred_labels, labels=labels_sorted)

    # Normalize by row (true label) for better visualization
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_sorted, yticklabels=labels_sorted,
                ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_xlabel('Predicted')
    ax1.set_ylabel('True')
    ax1.set_title('Confusion Matrix (Counts)')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0, fontsize=8)

    # Normalized
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=labels_sorted, yticklabels=labels_sorted,
                ax=ax2, cbar_kws={'shrink': 0.8})
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('True')
    ax2.set_title('Confusion Matrix (Normalized)')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=8)

    plt.suptitle('ForensicRoBERTa Classification Results', fontsize=14, fontweight='bold')
    plt.tight_layout()

    fig_path = os.path.join(RESULTS_DIR, 'fig_confusion_matrix.pdf')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {fig_path}")

✓ Saved: /content/drive/MyDrive/forensicnlp/results/stage2_classification/fig_confusion_matrix.pdf


## CELL 13: Model Comparison Bar Chart


WHAT THIS DOES:
Creates a bar chart comparing all models on accuracy and macro-F1.
Error bars show standard deviation across seeds.


In [17]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

model_labels = [s['Model'] for s in results_summary]
acc_means = [s['acc_mean'] for s in results_summary]
f1_means = [s['f1_mean'] for s in results_summary]
f1_stds = [s['f1_std'] for s in results_summary]

colors = ['#999999', '#4A90D9', '#E8833A', '#2EAE6D']

# Accuracy
bars1 = ax1.bar(model_labels, acc_means, color=colors, alpha=0.85)
ax1.set_ylabel('Accuracy')
ax1.set_title('Test Accuracy by Model')
ax1.set_ylim(0, 1.0)
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, acc_means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Macro-F1
bars2 = ax2.bar(model_labels, f1_means, yerr=f1_stds, color=colors,
                alpha=0.85, capsize=5)
ax2.set_ylabel('Macro-F1')
ax2.set_title('Test Macro-F1 by Model (mean ± std)')
ax2.set_ylim(0, 1.0)
ax2.grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, f1_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('ForensicNLP Classification Results', fontsize=14, fontweight='bold')
plt.tight_layout()

fig_path2 = os.path.join(RESULTS_DIR, 'fig_model_comparison.pdf')
plt.savefig(fig_path2, dpi=300, bbox_inches='tight')
plt.savefig(fig_path2.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fig_path2}")

✓ Saved: /content/drive/MyDrive/forensicnlp/results/stage2_classification/fig_model_comparison.pdf


## CELL 14: LaTeX Results Table


WHAT THIS DOES:
Generates a LaTeX table ready to paste into the paper.


In [18]:
print("=" * 60)
print("  LATEX TABLE (copy into paper)")
print("=" * 60)
print()

latex = r"""\begin{table}[t]
\centering
\caption{Classification Results (mean $\pm$ std over 3 seeds)}
\label{tab:classification}
\small
\renewcommand{\arraystretch}{1.15}
\begin{tabular}{@{}lcc@{}}
\toprule
\textbf{Model} & \textbf{Accuracy} & \textbf{Macro-F1} \\
\midrule
"""

for s in results_summary:
    model_name_tex = s['Model'].replace('&', r'\&')
    # Bold the best F1
    is_best = s['f1_mean'] == max(r['f1_mean'] for r in results_summary)

    if is_best:
        latex += f"\\textbf{{{model_name_tex}}} & \\textbf{{{s['Accuracy']}}} & \\textbf{{{s['Macro-F1']}}} \\\\\n"
    else:
        latex += f"{model_name_tex} & {s['Accuracy']} & {s['Macro-F1']} \\\\\n"

latex += r"""\bottomrule
\end{tabular}
\end{table}"""

print(latex)

  LATEX TABLE (copy into paper)

\begin{table}[t]
\centering
\caption{Classification Results (mean $\pm$ std over 3 seeds)}
\label{tab:classification}
\small
\renewcommand{\arraystretch}{1.15}
\begin{tabular}{@{}lcc@{}}
\toprule
\textbf{Model} & \textbf{Accuracy} & \textbf{Macro-F1} \\
\midrule
TF-IDF + LR & 0.8046 & 0.7506 \\
BERT & 0.8699 ± 0.0011 & 0.8196 ± 0.0070 \\
RoBERTa & 0.8686 ± 0.0055 & 0.8089 ± 0.0036 \\
\textbf{ForensicRoBERTa} & \textbf{0.8818 ± 0.0050} & \textbf{0.8387 ± 0.0108} \\
\bottomrule
\end{tabular}
\end{table}


## CELL 15: Save All Results


WHAT THIS DOES:
Saves everything to disk for the paper and future analysis.


In [19]:
# Save raw results
with open(os.path.join(RESULTS_DIR, 'all_results.json'), 'w') as f:
    # Remove numpy arrays for JSON serialization
    serializable = []
    for r in all_results:
        s = {k: v for k, v in r.items() if k != 'test_results'}
        serializable.append(s)
    json.dump(serializable, f, indent=2)

# Save summary
with open(os.path.join(RESULTS_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f"✓ Saved all_results.json")
print(f"✓ Saved results_summary.json")
print(f"✓ Saved figures (PDF + PNG)")
print(f"  Location: {RESULTS_DIR}/")

✓ Saved all_results.json
✓ Saved results_summary.json
✓ Saved figures (PDF + PNG)
  Location: /content/drive/MyDrive/forensicnlp/results/stage2_classification/


## CELL 16: Summary


WHAT THIS DOES:
Prints the key takeaways from Stage 2.


In [20]:
print("=" * 60)
print("  STAGE 2 COMPLETE — KEY FINDINGS")
print("=" * 60)

best_model = max(results_summary, key=lambda x: x['f1_mean'])
baseline = [s for s in results_summary if s['Model'] == 'TF-IDF + LR'][0]

print(f"""
BEST MODEL: {best_model['Model']}
  Macro-F1: {best_model['Macro-F1']}

vs BASELINE (TF-IDF + LR):
  Macro-F1: {baseline['Macro-F1']}
  Improvement: {(best_model['f1_mean'] - baseline['f1_mean'])*100:.2f} points

ForensicRoBERTa vs RoBERTa:
  Gain: {gain:+.2f} macro-F1 points
  Threshold met: {'YES (≥3 points)' if gain >= 3 else 'NO (< 3 points)'}

INTERPRETATION:
  → The results {'confirm' if gain > 0 else 'do not confirm'} RQ2:
    domain-adapted models {'do' if gain > 0 else 'do not'} outperform
    general models on forensic classification.
  → All transformer models {'outperform' if best_model['f1_mean'] > baseline['f1_mean'] else 'do not outperform'}
    the TF-IDF baseline, showing deep learning adds value.

FILES CREATED:
  {RESULTS_DIR}/
    ├── all_results.json
    ├── results_summary.json
    ├── fig_confusion_matrix.pdf/png
    └── fig_model_comparison.pdf/png

  {MODELS_DIR}/
    ├── forensic_roberta/          (domain-adapted model)
    └── ForensicRoBERTa_best/      (best fine-tuned checkpoint)

NEXT STEP: Run notebook 04 — RAG QA Experiments (Stage 3)
""")

  STAGE 2 COMPLETE — KEY FINDINGS

BEST MODEL: ForensicRoBERTa
  Macro-F1: 0.8387 ± 0.0108

vs BASELINE (TF-IDF + LR):
  Macro-F1: 0.7506
  Improvement: 8.81 points

ForensicRoBERTa vs RoBERTa:
  Gain: +2.98 macro-F1 points
  Threshold met: NO (< 3 points)

INTERPRETATION:
  → The results confirm RQ2: 
    domain-adapted models do outperform 
    general models on forensic classification.
  → All transformer models outperform 
    the TF-IDF baseline, showing deep learning adds value.

FILES CREATED:
  /content/drive/MyDrive/forensicnlp/results/stage2_classification/
    ├── all_results.json
    ├── results_summary.json
    ├── fig_confusion_matrix.pdf/png
    └── fig_model_comparison.pdf/png
  
  /content/drive/MyDrive/forensicnlp/models/
    ├── forensic_roberta/          (domain-adapted model)
    └── ForensicRoBERTa_best/      (best fine-tuned checkpoint)

NEXT STEP: Run notebook 04 — RAG QA Experiments (Stage 3)

